# 0.1 Overview

The representative population space is generated independently from multiple demographic datasets.

Although each attribute is statistically plausible, some attribute combinations are logically inconsistent.

This notebook applies a series of logical constraints to improve internal consistency before constructing behaviour profiles.

Input dataset:

- representative_population_space.parquet

Output:

- constrained_representative_population_space.parquet

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

### 2. Set Up File Paths

In [2]:
from pathlib import Path

project_root = Path.cwd().parents[1]

input_path = (
    project_root
    / "output"
    / "demographic_profiles"
)

output_path = (
    project_root
    / "output"
    / "demographic_profiles"
)

output_path.mkdir(
    parents=True,
    exist_ok=True
)

### 3. Load Dataset

In [3]:
population_space_file = (
    input_path
    / "representative_population_space.parquet"
)

population_space = pd.read_parquet(
    population_space_file
)

display(population_space.head())

print(population_space.shape)

,Age,Gender,Household Type,Household Count,Economic Activity Status,Economic Count,Industry,Industry Count,Ethnic Group,Ethnic Count,Distance travelled to work,Travel Distance Count,Car Ownership,Car Ownership Count
0,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,1 car or van in household,3139
1,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,2 cars or vans in household,3034
2,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,3 or more cars or vans in household,1723
3,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,10km to less than 30km,0,No cars or vans in household,1013
4,0-4,Female,Couple household: All children non-dependent,0,Does not apply,34378,"A Agriculture, forestry and fishing",0,"Asian, Asian British or Asian Welsh: Bangladeshi",833,30km to less than 60km,0,1 car or van in household,3139


(13099968, 14)


## 4. Age Rule

Economic activity is not applicable to profiles aged 0–4 or 5–11.

Profiles in these age groups are therefore retained only when:

`Economic Activity Status = Does not apply`

Other age groups are not modified by this rule.

In [4]:
YOUNG_CHILDREN = [
    "0-4",
    "5-11",
]

young_children = population_space[
    "Age"
].isin(YOUNG_CHILDREN)

age_rule_passed = (
    (young_children == False)
    |
    (
        population_space[
            "Economic Activity Status"
        ]
        == "Does not apply"
    )
)

profiles_before_age_rule = len(
    population_space
)

profiles_removed_by_age_rule = (
    age_rule_passed == False
).sum()

population_space = (
    population_space.loc[
        age_rule_passed
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Profiles before age rule:",
    f"{profiles_before_age_rule:,}",
)

print(
    "Profiles removed by age rule:",
    f"{profiles_removed_by_age_rule:,}",
)

print(
    "Profiles remaining:",
    f"{len(population_space):,}",
)

Profiles before age rule: 13,099,968
Profiles removed by age rule: 2,495,232
Profiles remaining: 10,604,736


In [5]:
young_children_profiles = population_space.loc[
    population_space["Age"].isin(
        YOUNG_CHILDREN
    )
]

age_rule_valid = (
    young_children_profiles[
        "Economic Activity Status"
    ]
    == "Does not apply"
).all()

assert age_rule_valid

print("Age rule validation: PASS")

display(
    young_children_profiles[
        [
            "Age",
            "Economic Activity Status",
            "Economic Count",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "Age",
            "Economic Activity Status",
        ]
    )
)

Age rule validation: PASS


,Age,Economic Activity Status,Economic Count
0,0-4,Does not apply,34378
207936,5-11,Does not apply,51349


## 5. Employment Rule

Industry and distance travelled to work are employment-related attributes.

Profiles classified as employed must have an applicable industry and work-distance category.

For profiles not classified as employed, both attributes must be recorded as `Does not apply`.

Profiles that do not satisfy these conditions are removed from the representative demographic profile space.

In [6]:
EMPLOYED_STATUSES = [
    (
        "Economically active "
        "(excluding full-time students): "
        "In employment"
    ),
    (
        "Economically active and a "
        "full-time student: In employment"
    ),
]

DOES_NOT_APPLY = "Does not apply"

is_employed = population_space[
    "Economic Activity Status"
].isin(EMPLOYED_STATUSES)

industry_applies = (
    population_space["Industry"]
    != DOES_NOT_APPLY
)

work_distance_applies = (
    population_space[
        "Distance travelled to work"
    ]
    != DOES_NOT_APPLY
)

employed_rule_passed = (
    is_employed
    & industry_applies
    & work_distance_applies
)

not_employed_rule_passed = (
    (is_employed == False)
    & (industry_applies == False)
    & (work_distance_applies == False)
)

employment_rule_passed = (
    employed_rule_passed
    | not_employed_rule_passed
)

profiles_before_employment_rule = len(
    population_space
)

profiles_removed_by_employment_rule = (
    employment_rule_passed == False
).sum()

population_space = (
    population_space.loc[
        employment_rule_passed
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Profiles before employment rule:",
    f"{profiles_before_employment_rule:,}",
)

print(
    "Profiles removed by employment rule:",
    f"{profiles_removed_by_employment_rule:,}",
)

print(
    "Profiles remaining:",
    f"{len(population_space):,}",
)

Profiles before employment rule: 10,604,736
Profiles removed by employment rule: 8,140,968
Profiles remaining: 2,463,768


In [7]:
is_employed = population_space[
    "Economic Activity Status"
].isin(EMPLOYED_STATUSES)

employed_profiles = population_space.loc[
    is_employed
]

not_employed_profiles = population_space.loc[
    is_employed == False
]

employed_industry_valid = (
    employed_profiles["Industry"]
    != DOES_NOT_APPLY
).all()

employed_distance_valid = (
    employed_profiles[
        "Distance travelled to work"
    ]
    != DOES_NOT_APPLY
).all()

not_employed_industry_valid = (
    not_employed_profiles["Industry"]
    == DOES_NOT_APPLY
).all()

not_employed_distance_valid = (
    not_employed_profiles[
        "Distance travelled to work"
    ]
    == DOES_NOT_APPLY
).all()

assert employed_industry_valid
assert employed_distance_valid
assert not_employed_industry_valid
assert not_employed_distance_valid

print("Employment rule validation: PASS")

print(
    "Employed profiles:",
    f"{len(employed_profiles):,}",
)

print(
    "Not-employed profiles:",
    f"{len(not_employed_profiles):,}",
)

Employment rule validation: PASS
Employed profiles: 2,413,152
Not-employed profiles: 50,616


## 6. Household-related Constraints
Household-related constraints are applied using the semantic definitions of the household categories. Unlike employment-related attributes, logically inconsistent household types cannot be replaced with a meaningful "Does not apply" category. Therefore, infeasible Age–Household Type combinations are removed from the representative population.

Three constraints are applied:

1. Household types in which all household members are aged 66 years and over are restricted to the older age groups.
2. Young children cannot be assigned to household types in which all children are non-dependent.
3. Young children cannot be assigned to couple households with no children.

## 6. Household Rules

Household categories must be compatible with the age of the representative profile.

Three rules are applied:

1. `Older family household` is restricted to the older age groups.
2. Profiles aged 0–4 or 5–11 cannot belong to households in which all children are non-dependent.
3. Profiles aged 0–4 or 5–11 cannot belong to a couple household with no children.

Profiles that do not satisfy these rules are removed.

In [8]:
OLDER_AGE_GROUPS = [
    "65-74",
    "75+",
]

OLDER_HOUSEHOLD = (
    "Older family household"
)

NON_DEPENDENT_CHILD_HOUSEHOLDS = [
    (
        "Couple household: "
        "All children non-dependent"
    ),
    (
        "Lone parent household: "
        "All children non-dependent"
    ),
]

NO_CHILDREN_HOUSEHOLDS = [
    "Couple household: No children",
]

required_household_categories = {
    OLDER_HOUSEHOLD,
    *NON_DEPENDENT_CHILD_HOUSEHOLDS,
    *NO_CHILDREN_HOUSEHOLDS,
}

available_household_categories = set(
    population_space[
        "Household Type"
    ].unique()
)

missing_household_categories = (
    required_household_categories
    - available_household_categories
)

if missing_household_categories:
    raise ValueError(
        "Missing household categories: "
        f"{sorted(missing_household_categories)}"
    )

print("Required household categories found.") 
# This allows that in the future, if we add more household categories, we don't have to update this code. We just need to ensure that the required categories are present.

Required household categories found.


In [9]:
# Apply household rules to the population pool
age = population_space["Age"]

household_type = population_space[
    "Household Type"
]

is_young_child = age.isin(
    YOUNG_CHILDREN
)

older_household_rule_passed = (
    (
        household_type
        != OLDER_HOUSEHOLD
    )
    |
    age.isin(OLDER_AGE_GROUPS)
)

non_dependent_children_rule_passed = (
    (is_young_child == False)
    |
    (
        household_type.isin(
            NON_DEPENDENT_CHILD_HOUSEHOLDS
        )
        == False
    )
)

no_children_household_rule_passed = (
    (is_young_child == False)
    |
    (
        household_type.isin(
            NO_CHILDREN_HOUSEHOLDS
        )
        == False
    )
)

household_rules_passed = (
    older_household_rule_passed
    & non_dependent_children_rule_passed
    & no_children_household_rule_passed
)

profiles_before_household_rules = len(
    population_space
)

profiles_removed_by_household_rules = (
    household_rules_passed == False
).sum()

population_space = (
    population_space.loc[
        household_rules_passed
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Profiles before household rules:",
    f"{profiles_before_household_rules:,}",
)

print(
    "Profiles removed by household rules:",
    f"{profiles_removed_by_household_rules:,}",
)

print(
    "Profiles remaining:",
    f"{len(population_space):,}",
)

Profiles before household rules: 2,463,768
Profiles removed by household rules: 196,536
Profiles remaining: 2,267,232


In [10]:
older_household_profiles = (
    population_space.loc[
        population_space[
            "Household Type"
        ]
        == OLDER_HOUSEHOLD
    ]
)

young_child_profiles = (
    population_space.loc[
        population_space["Age"].isin(
            YOUNG_CHILDREN
        )
    ]
)

older_household_age_valid = (
    older_household_profiles[
        "Age"
    ]
    .isin(OLDER_AGE_GROUPS)
    .all()
)

young_children_not_in_non_dependent_households = (
    young_child_profiles[
        "Household Type"
    ]
    .isin(
        NON_DEPENDENT_CHILD_HOUSEHOLDS
    )
    .any()
    == False
)

young_children_not_in_no_children_households = (
    young_child_profiles[
        "Household Type"
    ]
    .isin(
        NO_CHILDREN_HOUSEHOLDS
    )
    .any()
    == False
)

assert older_household_age_valid
assert (
    young_children_not_in_non_dependent_households
)
assert (
    young_children_not_in_no_children_households
)

print("Household rules validation: PASS")

Household rules validation: PASS


## 7. Final Validation

The final representative demographic profile space is validated against all age, employment, and household rules.

The validation confirms that:

- young children have no applicable economic activity;
- employment-related attributes agree with employment status;
- household type is compatible with age; and
- no duplicate demographic profiles remain.

In [11]:
young_children = population_space[
    "Age"
].isin(YOUNG_CHILDREN)

is_employed = population_space[
    "Economic Activity Status"
].isin(EMPLOYED_STATUSES)

is_older_household = (
    population_space["Household Type"]
    == OLDER_HOUSEHOLD
)

young_children_valid = (
    population_space.loc[
        young_children,
        "Economic Activity Status",
    ]
    == DOES_NOT_APPLY
).all()

employed_industry_valid = (
    population_space.loc[
        is_employed,
        "Industry",
    ]
    != DOES_NOT_APPLY
).all()

employed_distance_valid = (
    population_space.loc[
        is_employed,
        "Distance travelled to work",
    ]
    != DOES_NOT_APPLY
).all()

not_employed_industry_valid = (
    population_space.loc[
        is_employed == False,
        "Industry",
    ]
    == DOES_NOT_APPLY
).all()

not_employed_distance_valid = (
    population_space.loc[
        is_employed == False,
        "Distance travelled to work",
    ]
    == DOES_NOT_APPLY
).all()

older_household_valid = (
    population_space.loc[
        is_older_household,
        "Age",
    ]
    .isin(OLDER_AGE_GROUPS)
    .all()
)

young_children_household_valid = (
    population_space.loc[
        young_children,
        "Household Type",
    ]
    .isin(
        NON_DEPENDENT_CHILD_HOUSEHOLDS
        + NO_CHILDREN_HOUSEHOLDS
    )
    .any()
    == False
)

validation_results = {
    "Young children and economic activity": (
        young_children_valid
    ),
    "Employed profiles and industry": (
        employed_industry_valid
    ),
    "Employed profiles and work distance": (
        employed_distance_valid
    ),
    "Not-employed profiles and industry": (
        not_employed_industry_valid
    ),
    "Not-employed profiles and work distance": (
        not_employed_distance_valid
    ),
    "Older household and age": (
        older_household_valid
    ),
    "Young children and household type": (
        young_children_household_valid
    ),
}

validation_summary = pd.DataFrame(
    [
        {
            "Rule": rule,
            "Validation": (
                "PASS"
                if passed
                else "FAIL"
            ),
        }
        for rule, passed
        in validation_results.items()
    ]
)

display(validation_summary)

failed_rules = [
    rule
    for rule, passed
    in validation_results.items()
    if passed == False
]

if failed_rules:
    raise ValueError(
        "Final validation failed: "
        f"{failed_rules}"
    )

print("All representative population rules: PASS")

print(
    "Final population shape:",
    population_space.shape,
)

,Rule,Validation
0,Young children and economic activity,PASS
1,Employed profiles and industry,PASS
2,Employed profiles and work distance,PASS
3,Not-employed profiles and industry,PASS
4,Not-employed profiles and work distance,PASS
5,Older household and age,PASS
6,Young children and household type,PASS


All representative population rules: PASS
Final population shape: (2267232, 14)


In [12]:
PROFILE_ATTRIBUTES = [
    "Age",
    "Gender",
    "Household Type",
    "Economic Activity Status",
    "Industry",
    "Ethnic Group",
    "Distance travelled to work",
    "Car Ownership",
]

duplicate_profiles = (
    population_space
    .duplicated(
        subset=PROFILE_ATTRIBUTES
    )
    .sum()
)

complete_row_duplicates = (
    population_space
    .duplicated()
    .sum()
)

duplicate_summary = pd.DataFrame(
    [
        {
            "Duplicate Type": (
                "Demographic profile duplicates"
            ),
            "Count": duplicate_profiles,
        },
        {
            "Duplicate Type": (
                "Complete row duplicates"
            ),
            "Count": complete_row_duplicates,
        },
    ]
)

display(duplicate_summary)

if duplicate_profiles > 0:
    raise ValueError(
        "Duplicate demographic profiles found: "
        f"{duplicate_profiles:,}"
    )

if complete_row_duplicates > 0:
    raise ValueError(
        "Completely identical rows found: "
        f"{complete_row_duplicates:,}"
    )

print("Duplicate validation: PASS")

,Duplicate Type,Count
0,Demographic profile duplicates,0
1,Complete row duplicates,0


Duplicate validation: PASS


## 8. Export

The validated demographic profile space is exported for integration with the Behavioural Evidence Layer.

Each row represents one logically feasible demographic combination. It does not represent an individual synthetic person or a known joint population weight.

Supporting Census counts are retained for provenance and source validation, but they are not treated as individual behavioural attributes.

In [13]:
output_path.mkdir(
    parents=True,
    exist_ok=True,
)

constrained_population_space = (
    output_path
    / (
        "constrained_representative_"
        "population_space.parquet"
    )
)

population_space.to_parquet(
    constrained_population_space,
    index=False,
)

print(
    "Constrained representative population "
    "exported to:"
)

print(constrained_population_space)

print(
    "Exported shape:",
    population_space.shape,
)

Constrained representative population exported to:
c:\Users\user\OneDrive - University of Stirling\Desktop\dissertation\mobility-modelling-framework\output\demographic_profiles\constrained_representative_population_space.parquet
Exported shape: (2267232, 14)
